# Industrial Surface Defect Detection Using Transfer Learning and Explainable AI

**Course:** Deep Learning (Unit 5: Transfer Learning & Explainable AI)  
**Architecture:** MobileNetV2 Pretrained Backbone + Grad-CAM  
**Dataset:** MVTec Anomaly Detection (`bottle` category)  

---

## Notebook Overview
This interactive notebook demonstrates the complete end-to-end workflow:
1. **Environment & Hardware Inspection**
2. **Dataset Acquisition & Stratified Splitting**
3. **Experiment 1: Feature Extraction (Frozen MobileNetV2 Backbone)**
4. **Experiment 2: Fine-Tuning (Selective Unfreezing of Upper Layers)**
5. **Comparative Evaluation on Held-Out Test Set**
6. **Explainable AI (Grad-CAM) Visual Localization**

In [ ]:
# Step 1: Environment and Path Setup
import os
import sys
from pathlib import Path

# Ensure project root is in sys.path
project_root = Path.cwd().resolve()
if str(project_root) not in sys.path:
    sys.path.insert(0, str(project_root))
if str(project_root.parent) not in sys.path:
    sys.path.insert(0, str(project_root.parent))

import config
import tensorflow as tf
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from PIL import Image

print(f"TensorFlow Version: {tf.__version__}")
print(f"Devices Available: {tf.config.list_physical_devices()}")
print(f"Project Workspace: {config.BASE_DIR}")

## 2. Dataset Exploration & Preprocessing
We load the MVTec AD `bottle` dataset and inspect samples from both the **Normal (Non-Defective)** and **Defective (Anomalous)** classes.

In [ ]:
from src.dataset import prepare_splits, get_dataset_summary

# Prepare or load verified dataset splits
splits_meta = prepare_splits(category="bottle", force=False)

print("Dataset Summary:")
print(f"Total Images: {splits_meta['raw_summary']['total_valid_samples']}")
print(f"Normal: {splits_meta['raw_summary']['normal_samples']} | Defective: {splits_meta['raw_summary']['defective_samples']}")
for split_name, stats in splits_meta['splits'].items():
    print(f"  {split_name.upper()}: {stats['total']} images (Normal: {stats['normal']}, Defective: {stats['defective']})")

In [ ]:
# Visualize Sample Images
test_dir = config.PROCESSED_DATA_DIR / "bottle" / "test"
sample_norm = list((test_dir / "normal").glob("*.png"))[0]
sample_def = list((test_dir / "defective").glob("*.png"))[0]

fig, axes = plt.subplots(1, 2, figsize=(10, 5))
axes[0].imshow(Image.open(sample_norm))
axes[0].set_title("Sample Normal Bottle (Class 0)", fontweight="bold")
axes[0].axis("off")

axes[1].imshow(Image.open(sample_def))
axes[1].set_title("Sample Defective Bottle (Class 1)", fontweight="bold")
axes[1].axis("off")

plt.tight_layout()
plt.show()

## 3. Experiment 1: Feature Extraction (Frozen Backbone)
In Feature Extraction, the pretrained MobileNetV2 base is frozen (`trainable = False`), and only the attached binary classification head is trained.

In [ ]:
from src.train_feature_extraction import build_feature_extraction_model

model_fe = build_feature_extraction_model()
print(f"Total Model Parameters: {model_fe.count_params():,}")
trainable_p = sum(tf.keras.backend.count_params(w) for w in model_fe.trainable_weights)
non_trainable_p = sum(tf.keras.backend.count_params(w) for w in model_fe.non_trainable_weights)
print(f"Trainable Parameters (Head only): {trainable_p:,}")
print(f"Non-Trainable Parameters (Frozen Base): {non_trainable_p:,}")

In [ ]:
# Display Feature Extraction Training Curves
if config.FE_CURVES_PATH.exists():
    img_curves = Image.open(config.FE_CURVES_PATH)
    plt.figure(figsize=(14, 5))
    plt.imshow(img_curves)
    plt.axis("off")
    plt.title("Feature Extraction Training History", fontsize=14, fontweight="bold")
    plt.show()

## 4. Experiment 2: Fine-Tuning (Selective Layer Unfreezing)
Top layers of MobileNetV2 from layer index 100 onwards are unfrozen, while lower layers and Batch Normalization layers remain frozen. Training proceeds with a lower learning rate ($10^{-5}$).

In [ ]:
if config.FT_CURVES_PATH.exists():
    img_ft_curves = Image.open(config.FT_CURVES_PATH)
    plt.figure(figsize=(14, 5))
    plt.imshow(img_ft_curves)
    plt.axis("off")
    plt.title("Fine-Tuning Training History", fontsize=14, fontweight="bold")
    plt.show()

## 5. Comparative Evaluation on Held-Out Test Set
We evaluate both models on the exact same held-out test set ($N=44$) to assess performance without data leakage.

In [ ]:
if config.MODEL_COMPARISON_CSV.exists():
    df_comparison = pd.read_csv(config.MODEL_COMPARISON_CSV)
    print("Held-Out Test Set Comparison:")
    display(df_comparison)
else:
    from src.evaluate import evaluate_models
    df_comparison = evaluate_models()
    display(df_comparison)

In [ ]:
# Display Confusion Matrices
fig, axes = plt.subplots(1, 2, figsize=(12, 5))
if config.FE_CONF_MATRIX_PATH.exists():
    axes[0].imshow(Image.open(config.FE_CONF_MATRIX_PATH))
    axes[0].set_title("Feature Extraction Confusion Matrix", fontweight="bold")
    axes[0].axis("off")

if config.FT_CONF_MATRIX_PATH.exists():
    axes[1].imshow(Image.open(config.FT_CONF_MATRIX_PATH))
    axes[1].set_title("Fine-Tuning Confusion Matrix", fontweight="bold")
    axes[1].axis("off")

plt.tight_layout()
plt.show()

## 6. Explainable AI: Grad-CAM Feature Attribution
We generate visual explanations for predictions using Grad-CAM on layer `out_relu` of MobileNetV2.

In [ ]:
from src.gradcam import explain_prediction

# Run Grad-CAM on sample defective bottle
defect_sample_path = list((config.PROCESSED_DATA_DIR / "bottle" / "test" / "defective").glob("*.png"))[0]
explanation = explain_prediction(defect_sample_path, model_path=config.FINETUNED_MODEL_PATH)

fig, axes = plt.subplots(1, 3, figsize=(15, 5))
axes[0].imshow(explanation["display_image"])
axes[0].set_title("Input Image (RGB)", fontsize=12, fontweight="bold")
axes[0].axis("off")

axes[1].imshow(explanation["heatmap"], cmap="jet")
axes[1].set_title("Grad-CAM Heatmap (out_relu)", fontsize=12, fontweight="bold")
axes[1].axis("off")

axes[2].imshow(explanation["overlay_image"])
axes[2].set_title(f"Overlay ({explanation['prediction']['display_label']}: {explanation['prediction']['confidence_percentage']})", fontsize=12, fontweight="bold")
axes[2].axis("off")

plt.tight_layout()
plt.show()

print("Explanation Interpretation:")
print(explanation["interpretation"])

## Conclusion & Key Takeaways
1. **Transfer Learning Effectiveness:** Pretrained MobileNetV2 achieved **97.73% Accuracy** with both feature extraction and fine-tuning.
2. **Operational Priority:** Fine-tuning achieved **100% Recall** on defects ($9/9$), eliminating dangerous false negative defect escapes.
3. **Explainability:** Grad-CAM precisely pinpoints the physical crack/contamination region, providing actionable transparency for industrial quality assurance.